# LimiX Classification Example

This notebook demonstrates how to use the FAIM Python SDK's **TabularClient** with **LimiX** for tabular classification tasks.

[LimiX](https://github.com/limix-ldm/LimiX) is an open-source foundation model for tabular machine learning that supports both classification and regression.

To get an API key, go to the FAIM website: https://faim.it.com/api-keys

## Setup

Install dependencies and import required libraries.

In [30]:
import os

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from faim_sdk import LimiXPredictRequest, TabularClient

## Load and Prepare Data

Load the breast cancer classification dataset from scikit-learn.

In [31]:
# Load breast cancer dataset
X, y = load_breast_cancer(return_X_y=True)

# Split with 50/50 train-test split for demonstration
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Convert to float32 for API
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Classes: {np.unique(y_train)}")

Training set size: (426, 30)
Test set size: (143, 30)
Number of features: 30
Classes: [0. 1.]


## Initialize TabularClient

Create a client to interact with the LimiX model.

In [32]:
# Initialize the client
client = TabularClient(
    base_url="https://api.faim.it.com",
    api_key=os.environ.get("FAIM_API_KEY"),  # Replace with your actual API key
    timeout=120.0,
)

print("TabularClient initialized!")

TabularClient initialized!


## Create Classification Request

Prepare a LimiX classification request.

In [33]:
# Create a LimiX classification request
request = LimiXPredictRequest(
    X_train=X_train, y_train=y_train, X_test=X_test, task_type="Classification",
)

print("Request prepared:")
print(f"  X_train shape: {request.X_train.shape}")
print(f"  X_test shape: {request.X_test.shape}")
print(f"  Task type: {request.task_type}")

Request prepared:
  X_train shape: (426, 30)
  X_test shape: (143, 30)
  Task type: Classification


## Make Predictions

Send the request to LimiX and get classification predictions.

In [34]:
try:
    # Make predictions
    response = client.predict(request)

    print(f"Predictions shape: {response.predictions.shape}")
    print(f"First 10 predictions: {response.predictions[:10]}")

    if response.probabilities is not None:
        print(f"\nClass probabilities shape: {response.probabilities.shape}")
        print(f"First 3 samples probabilities:\n{response.probabilities[:3]}")

    print("\nMetadata:")
    for key, value in response.metadata.items():
        if key == 'cost_amount':
            value = float(value)/1e6
        print(f"  {key}: {value}")

except Exception as e:
    print(f"Error: {e}")

Predictions shape: (143,)
First 10 predictions: [1 0 0 1 1 0 0 0 1 1]

Class probabilities shape: (143, 2)
First 3 samples probabilities:
[[9.9312373e-02 9.0068763e-01]
 [9.9999857e-01 1.4161864e-06]
 [9.9999756e-01 2.4666476e-06]]

Metadata:
  model_name: limix
  model_version: 1
  transaction_id: 98571ccf-89dd-4363-92f5-16f27af90a8d
  cost_amount: 0.086065
  cost_currency: USD
  token_count: 17213


## Evaluate Results

Calculate classification metrics.

In [35]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

try:
    y_pred = response.predictions.astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("Classification Metrics:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
except NameError:
    print("Run prediction cell first to evaluate metrics.")

Classification Metrics:
  Accuracy:  0.9860
  Precision: 0.9888
  Recall:    0.9888
  F1-Score:  0.9888
